In [1]:
from dotenv import load_dotenv

load_dotenv()

True

## Summarize messages

In [2]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents.middleware import SummarizationMiddleware

agent = create_agent(
    model="google_genai:gemini-3.1-flash-lite",
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model="google_genai:gemini-3.1-flash-lite",
            trigger=("tokens", 100),
            keep=("messages", 1)
        )
    ],
)

In [3]:
from langchain.messages import HumanMessage, AIMessage
from pprint import pprint

response = agent.invoke(
    {"messages": [
        HumanMessage(content="What is the capital of the moon?"),
        AIMessage(content="The capital of the moon is Lunapolis."),
        HumanMessage(content="What is the weather in Lunapolis?"),
        AIMessage(content="Skies are clear, with a high of 120C and a low of -100C."),
        HumanMessage(content="How many cheese miners live in Lunapolis?"),
        AIMessage(content="There are 100,000 cheese miners living in Lunapolis."),
        HumanMessage(content="Do you think the cheese miners' union will strike?"),
        AIMessage(content="Yes, because they are unhappy with the new president."),
        HumanMessage(content="If you were Lunapolis' new president how would you respond to the cheese miners' union?"),
        ]},
    {"configurable": {"thread_id": "1"}}
)

pprint(response)

{'messages': [HumanMessage(content="Here is a summary of the conversation to date:\n\n## SESSION INTENT\n\nThe user is inquiring about fictional details regarding the moon, specifically the city of Lunapolis, its climate, demographics, and labor relations.\n\n## SUMMARY\n\n- The moon’s capital is identified as Lunapolis.\n- Environmental conditions: Clear skies, temperature range between 120°C and -100°C.\n- Population: 100,000 cheese miners.\n- Current status: The cheese miners' union is expected to strike due to dissatisfaction with the current president.\n\n## ARTIFACTS\n\nNone.\n\n## NEXT STEPS\n\n- Await further inquiries from the user regarding the fictional state of Lunapolis or other related topics.", additional_kwargs={'lc_source': 'summarization'}, response_metadata={}, id='f792bcb2-fe3a-45be-acc0-a7015762a90d'),
              HumanMessage(content="If you were Lunapolis' new president how would you respond to the cheese miners' union?", additional_kwargs={}, response_metadata

In [4]:
print(response["messages"][0].content)

Here is a summary of the conversation to date:

## SESSION INTENT

The user is inquiring about fictional details regarding the moon, specifically the city of Lunapolis, its climate, demographics, and labor relations.

## SUMMARY

- The moon’s capital is identified as Lunapolis.
- Environmental conditions: Clear skies, temperature range between 120°C and -100°C.
- Population: 100,000 cheese miners.
- Current status: The cheese miners' union is expected to strike due to dissatisfaction with the current president.

## ARTIFACTS

None.

## NEXT STEPS

- Await further inquiries from the user regarding the fictional state of Lunapolis or other related topics.


## Trim/delete messages

In [5]:
from typing import Any
from langchain.agents import AgentState
from langchain.messages import RemoveMessage
from langgraph.runtime import Runtime
from langchain.agents.middleware import before_agent
from langchain.messages import ToolMessage

@before_agent
def trim_messages(state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
    """Remove all the tool messages from the state"""
    messages = state["messages"]

    tool_messages = [m for m in messages if isinstance(m, ToolMessage)]
    
    return {"messages": [RemoveMessage(id=m.id) for m in tool_messages]}

In [6]:
agent = create_agent(
    model="google_genai:gemini-3.1-flash-lite",
    checkpointer=InMemorySaver(),
    middleware=[trim_messages],
)

In [7]:
response = agent.invoke(
    {"messages": [
        HumanMessage(content="My device won't turn on. What should I do?"),
        ToolMessage(content="blorp-x7 initiating diagnostic ping…", tool_call_id="1"),
        AIMessage(content="Is the device plugged in and turned on?"),
        HumanMessage(content="Yes, it's plugged in and turned on."),
        ToolMessage(content="temp=42C voltage=2.9v … greeble complete.", tool_call_id="2"),
        AIMessage(content="Is the device showing any lights or indicators?"),
        HumanMessage(content="What's the temperature of the device?")
        ]},
    {"configurable": {"thread_id": "2"}}
)

pprint(response)

{'messages': [HumanMessage(content="My device won't turn on. What should I do?", additional_kwargs={}, response_metadata={}, id='5bf98abe-20ea-4a5c-a225-eaf802d43cd0'),
              AIMessage(content='Is the device plugged in and turned on?', additional_kwargs={}, response_metadata={}, id='7af1c11b-269a-4eea-a5ec-2d02a946bad4', tool_calls=[], invalid_tool_calls=[]),
              HumanMessage(content="Yes, it's plugged in and turned on.", additional_kwargs={}, response_metadata={}, id='b74741dc-521f-4822-8d16-cc5b52397c0d'),
              AIMessage(content='Is the device showing any lights or indicators?', additional_kwargs={}, response_metadata={}, id='14266dca-8c2e-443f-bb9e-6d7bb88f0cac', tool_calls=[], invalid_tool_calls=[]),
              HumanMessage(content="What's the temperature of the device?", additional_kwargs={}, response_metadata={}, id='20510ff1-c2e5-42b5-b439-a938f96e08bd'),
              AIMessage(content=[{'type': 'text', 'text': 'That is an excellent question to ask

In [8]:
print(response["messages"][-1].content)

[{'type': 'text', 'text': 'That is an excellent question to ask. If a device is overheating, it may have a safety feature that prevents it from turning on to protect the internal components from damage.\n\n**Please check the following:**\n\n1.  **Feel the device:** Is it hot to the touch in any specific area?\n2.  **Look for obstructions:** Are the vents (if it has them) blocked by dust, fabric, or a surface like a blanket or rug?\n3.  **Recent usage:** Was it being used heavily or left in a hot environment (like a car or in direct sunlight) just before it stopped working?\n\n**If it feels unusually hot:**\n*   **Unplug it immediately** to stop any power flow.\n*   **Let it cool down** for at least 30–60 minutes in a cool, well-ventilated area.\n*   **Do not** put it in a freezer or fridge, as the condensation can cause internal water damage.\n\n**Once it has cooled down completely:**\n*   Try plugging it back in and attempting to power it on again.\n\n**If it is not hot:**\n*   Please